# In-silico simulation

In [ ]:
import os
os.chdir("./in_silico_simulation")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-


import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import adjusted_rand_score, confusion_matrix
from scipy.optimize import linear_sum_assignment

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
BASE_DIR = Path("Type your path here")

SETTINGS = [
    {"K": 3, "reactive_fraction": 0.3, "dropout": 0.02, "mutation_p": 0.05, "background_p": 0.001, "rep": 10},
    {"K": 2, "reactive_fraction": 0.3, "dropout": 0.02, "mutation_p": 0.05, "background_p": 0.001, "rep": 10},
    {"K": 3, "reactive_fraction": 0.3, "dropout": 0.02, "mutation_p": 0.02, "background_p": 0.001, "rep": 10},
    {"K": 2, "reactive_fraction": 0.3, "dropout": 0.02, "mutation_p": 0.02, "background_p": 0.001, "rep": 10},
]

MODES = ["uniform", "mild_skew", "extreme"]
seq_lengths = [600, 900, 1500]
num_sequences = 3
base_probs = [0.3, 0.2, 0.3, 0.2]
depth_per_base = 8000

from varisem import VARISEM
from EM import PureEM


# --------------------------------------------------
# Helpers
# --------------------------------------------------
def load_mu_csv(path):
    df = pd.read_csv(path)
    df = df.select_dtypes(include=["number"])
    return df.values.astype(float)


def load_pi_csv(path):
    df = pd.read_csv(path)
    df = df.select_dtypes(include=["number"])
    return df.values.flatten().astype(float)


def load_cluster_labels(path):
    df = pd.read_csv(path)
    for c in df.columns:
        if "cluster" in c.lower():
            return df[c].astype(int).values
    return df.iloc[:, -1].astype(int).values


# --------------------------------------------------
# Compute KL(all)
# --------------------------------------------------
def compute_KL_all(mu_true, mu_est):
    eps = 1e-10

    def KL(p, q):
        p = np.clip(p, eps, 1 - eps)
        q = np.clip(q, eps, 1 - eps)
        return p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q))

    K, D = mu_true.shape
    KL_matrix_all = np.zeros((K, D))

    for k in range(K):
        KL_matrix_all[k] = KL(mu_true[k], mu_est[k])

    KL_all_mean = KL_matrix_all.mean()
    return KL_all_mean, KL_matrix_all


# --------------------------------------------------
# Hungarian alignment
# --------------------------------------------------
def hungarian_align(Z_true, Z_pred, K, pi_inf, mu_inf):
    cm = confusion_matrix(Z_true, Z_pred, labels=range(K))
    row, col = linear_sum_assignment(-cm)

    mapping = dict(zip(col, row))
    Z_aligned = np.array([mapping[z] for z in Z_pred])

    return Z_aligned, pi_inf[col], mu_inf[col]


# --------------------------------------------------
# DRACO-style Simulation
# --------------------------------------------------
def generate_in_silico_data(seed, K, reactive_fraction, dropout,
                             mutation_p, background_p, out_dir, mode):
    """
    DRACO-style synthetic dataset:
    - Each cluster has a continuous high-reactivity block (AC-only sites)
    - Different clusters occupy non-overlapping AC intervals
    """
    np.random.seed(seed)
    os.makedirs(out_dir, exist_ok=True)

    # True cluster proportions π
    if mode == "uniform":
        pi_true = np.ones(K) / K
    elif mode == "mild_skew":
        base = np.linspace(1.5, 1.0, K)
        pi_true = base / base.sum()
    elif mode == "extreme":
        base = np.linspace(3.0, 0.5, K)
        pi_true = base / base.sum()
    else:
        raise ValueError(f"Unknown mode: {mode}")

    # Generate sequence
    L = np.random.choice(seq_lengths)
    seq = np.random.choice(["A", "C", "T", "G"], L, p=base_probs)
    ac_positions = np.where(np.isin(seq, ["A", "C"]))[0]
    ac_len = len(ac_positions)

    # Block size
    block_size = int(ac_len * reactive_fraction)

    mu_true = np.zeros((K, L))
    union_mask = np.zeros(L, dtype=bool)

    # Split AC positions into K non-overlapping intervals
    split_points = np.linspace(0, ac_len, K + 1).astype(int)

    for k in range(K):
        start = split_points[k]
        end = split_points[k + 1]

        # Generate the continuous block within the interval
        if end - start < block_size:
            subset = np.random.choice(ac_positions, block_size, replace=False)
        else:
            local_AC = ac_positions[start:end]
            if len(local_AC) > block_size:
                start_idx = np.random.randint(0, len(local_AC) - block_size)
                subset = local_AC[start_idx:start_idx + block_size]
            else:
                subset = local_AC

        mask_k = np.zeros(L, dtype=bool)
        mask_k[subset] = True
        union_mask |= mask_k

        mu_true[k, mask_k] = mutation_p
        mu_true[k, ~mask_k] = background_p

    # Reads
    N = depth_per_base
    Z_true = np.random.choice(K, N, p=pi_true)

    X = np.zeros((N, L), dtype=int)
    for n in range(N):
        X[n] = np.random.binomial(1, mu_true[Z_true[n]])

    # Dropout
    X[np.random.rand(*X.shape) < dropout] = 0

    # Save
    df = pd.DataFrame(X, columns=[f"Pos_{i+1}_{seq[i]}" for i in range(L)])
    df["Cluster_true"] = Z_true
    df.to_csv(out_dir / "Seq1_sim.csv", index=False)

    np.save(out_dir / "mu_true_Seq1.npy", mu_true)
    np.save(out_dir / "pi_true_Seq1.npy", pi_true)
    np.save(out_dir / "Z_true_Seq1.npy", Z_true)
    np.save(out_dir / "reactive_mask.npy", union_mask)

    return pi_true


# --------------------------------------------------
# MAIN
# --------------------------------------------------
summary = []
performance_rows = []

for setting in SETTINGS:
    K = setting["K"]

    for mode in MODES:
        tag = f"K{K}-react{setting['reactive_fraction']}-drop{setting['dropout']}-mut{setting['mutation_p']}-{mode}"
        print(f"\n=== Running Setting {tag} ===")

        metric_records = {"VARISEM": [], "DREEM": []}

        for rep in range(1, setting["rep"] + 1):
            folder = BASE_DIR / f"sim-{tag}-rep{rep}"
            pi_true = generate_in_silico_data(
                1008 + rep,
                K,
                setting["reactive_fraction"],
                setting["dropout"],
                setting["mutation_p"],
                setting["background_p"],
                folder,
                mode
            )

            df = pd.read_csv(folder / "Seq1_sim.csv")
            X = df.drop(columns=["Cluster_true"]).values
            Z_true = df["Cluster_true"].values
            mu_true = np.load(folder / "mu_true_Seq1.npy")

            # =============================
            # Run VARISEM
            # =============================
            vi = VARISEM(X, K=K, max_iter=300)
            vi.fit()
            vi.save_results(folder / "pi_VI.csv", folder / "mu_VI.csv", folder / "Z_VI.csv")

            mu_vi = load_mu_csv(folder / "mu_VI.csv")
            pi_vi = load_pi_csv(folder / "pi_VI.csv")
            Z_vi = load_cluster_labels(folder / "Z_VI.csv")

            Z_a, pi_a, mu_a = hungarian_align(Z_true, Z_vi, K, pi_vi, mu_vi)

            KL_all, KL_mat_all = compute_KL_all(mu_true, mu_a)
            np.savetxt(folder / "KL_all_VI.csv", KL_mat_all, delimiter=",")

            metrics_vi = {
                "ARI": adjusted_rand_score(Z_true, Z_a),
                "MAE(π)": np.abs(pi_true - pi_a).mean(),
                "KL_all": KL_all,
            }
            metric_records["VARISEM"].append(metrics_vi)

            performance_rows.append({
                "Setting": tag,
                "Method": "VARISEM",
                "Rep": rep,
                **metrics_vi,
            })

            # =============================
            # Run EM
            # =============================
            em = PureEM(X, K=K, MIN_ITS=30)
            em.fit()
            em.save_results(folder / "pi_EM.csv", folder / "mu_EM.csv", folder / "Z_EM.csv")

            mu_em = load_mu_csv(folder / "mu_EM.csv")
            pi_em = load_pi_csv(folder / "pi_EM.csv")
            Z_em = load_cluster_labels(folder / "Z_EM.csv")

            Z_a2, pi_a2, mu_a2 = hungarian_align(Z_true, Z_em, K, pi_em, mu_em)

            KL_all2, KL_mat_all2 = compute_KL_all(mu_true, mu_a2)
            np.savetxt(folder / "KL_all_EM.csv", KL_mat_all2, delimiter=",")

            metrics_em = {
                "ARI": adjusted_rand_score(Z_true, Z_a2),
                "MAE(π)": np.abs(pi_true - pi_a2).mean(),
                "KL_all": KL_all2,
            }
            metric_records["DREEM"].append(metrics_em)

            performance_rows.append({
                "Setting": tag,
                "Method": "DREEM",
                "Rep": rep,
                **metrics_em,
            })

        # Summary table
        for m in ["VARISEM", "DREEM"]:
            dfm = pd.DataFrame(metric_records[m])
            summary.append({
                "Setting": tag,
                "Method": m,
                "K": K,
                "Mode": mode,
                **{col: f"{dfm[col].mean():.3f} ± {dfm[col].std():.3f}" for col in dfm.columns},
            })

pd.DataFrame(summary).to_csv(BASE_DIR / "summary.csv", index=False)
pd.DataFrame(performance_rows).to_csv(BASE_DIR / "performance.csv", index=False)
